# M17 v2 — OWL Stage 2: Incremental Class Incorporation & Forgetting Curve

**Model ID:** M17  
**Model Name:** OWL Stage 2 + Forgetting Curve  
**Member:** B — Disease Diagnosis & Staged Open-World Learning Lead  
**Project:** OWMTL — Cluster-Aware Open-World Multi-Task Learning for Respiratory Sound and Disease Diagnosis  
**Requires:** M2 Backbone (`best_model.pth`), M13 Prototypical Disease Head (`best_model.pth`), Real ICBHI Audio  

---

### The Novelty: Staged Open-World Learning (3 Stages)
| Stage | What Happens | Models |
|-------|-------------|--------|
| **Stage 0** | Train on Known classes (COPD, Healthy, URTI) | M2 backbone + M13 disease head |
| **Stage 1** | Detect Unknowns via cross-task disagreement | M15 consistency scorer |
| **Stage 2** | Incorporate a newly-labeled Unknown (Pneumonia) and measure forgetting | **M17 (this notebook)** |

### v2 Core Improvements Over v1
1. **REAL ICBHI Audio:** Loads actual `.wav` recordings (no synthetic `torch.randn`).
2. **Prototypical Network Expansion:** Adds Pneumonia as a 4th prototype — no classifier head surgery needed.
3. **Patient-Independent Replay Buffer:** Mixes old known-class patient cycles with new Pneumonia cycles.
4. **Per-Epoch Forgetting Curve:** Tracks Stage 0 retention accuracy at every training epoch.
5. **§4 Compliant `results_M17.json`** with full metrics suite and forgetting analysis.


## Section 1: Setup & Dependencies


In [ ]:
# ============================================================
# Section 1: Environment Setup & Dependencies
# ============================================================
import os
import sys
import re
import json
import math
import time
import glob
import copy
import random
import warnings
import datetime
import zipfile
import io
import shutil
import base64
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, ConcatDataset

from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    confusion_matrix
)

warnings.filterwarnings('ignore')

# ---- Reproducibility ----
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
GPU_NAME = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'

print(f'Device:  {DEVICE} ({GPU_NAME})')
print(f'PyTorch: {torch.__version__}')
print(f'Python:  {sys.version.split()[0]}')

plt.rcParams.update({'figure.dpi': 150, 'savefig.dpi': 150, 'font.size': 11})
sns.set_style('whitegrid')


## Section 2: Configuration & Path Resolution (Kaggle & Colab)


In [ ]:
# ============================================================
# Section 2: Configuration & Path Resolution (Kaggle & Colab)
# ============================================================

# ---- Auto-detect Platform ----
if os.path.exists('/kaggle'):
    PLATFORM = 'Kaggle'
    BASE_DIR = '/kaggle/working'
elif os.path.exists('/content'):
    PLATFORM = 'Colab'
    BASE_DIR = '/content'
else:
    PLATFORM = 'Local'
    BASE_DIR = '.'

print(f'Platform: {PLATFORM}')

# ---- Google Drive Mount (Colab) ----
DRIVE_DIR = None
if PLATFORM == 'Colab':
    try:
        from google.colab import drive
        drive.mount('/content/drive', force_remount=False)
        DRIVE_DIR = '/content/drive/MyDrive/OWMTL/M17'
        os.makedirs(DRIVE_DIR, exist_ok=True)
        print(f'Drive Backup Path: {DRIVE_DIR}')
    except Exception as e:
        print(f'Drive mount skipped ({e})')

# ---- ICBHI Dataset Path Resolution ----
POSSIBLE_ROOTS = [
    '/kaggle/input/respiratory-sound-database/Respiratory_Sound_Database/Respiratory_Sound_Database/audio_and_txt_files',
    '/kaggle/input/respiratory-sound-database/audio_and_txt_files',
    '/kaggle/input/respiratory-sound-database/Respiratory_Sound_Database/audio_and_txt_files',
    '/kaggle/input/icbhi-2017-respiratory-sound-database/audio_and_txt_files',
    '/kaggle/input/vbookshelf/respiratory-sound-database/Respiratory_Sound_Database/Respiratory_Sound_Database/audio_and_txt_files',
    '/content/Respiratory_Sound_Database/Respiratory_Sound_Database/audio_and_txt_files',
    '/content/drive/MyDrive/respiratory-sound-database/audio_and_txt_files',
    '/content/drive/MyDrive/OWMTL/data/audio_and_txt_files',
    './data/audio_and_txt_files',
]
DATA_ROOT = next((p for p in POSSIBLE_ROOTS if os.path.exists(p)), None)

if DATA_ROOT is None and os.path.exists('/kaggle/input'):
    for root, dirs, files in os.walk('/kaggle/input'):
        if any(f.endswith('.wav') for f in files) and any(f.endswith('.txt') for f in files):
            DATA_ROOT = root
            print(f'Dynamic Kaggle resolution: {DATA_ROOT}')
            break

# ---- Colab Kaggle auto-download ----
if DATA_ROOT is None and PLATFORM == 'Colab':
    print('\n📥 ICBHI dataset not found. Checking Kaggle credentials...')
    drive_kjson = '/content/drive/MyDrive/kaggle.json'
    if os.path.exists(drive_kjson):
        os.system('mkdir -p ~/.kaggle && cp /content/drive/MyDrive/kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json')
    elif not os.path.exists(os.path.expanduser('~/.kaggle/kaggle.json')):
        try:
            from google.colab import files
            print('Please upload kaggle.json:')
            uploaded = files.upload()
            if 'kaggle.json' in uploaded:
                os.system('mkdir -p ~/.kaggle && mv kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json')
        except Exception: pass

    if os.path.exists(os.path.expanduser('~/.kaggle/kaggle.json')):
        os.system('pip install -q kaggle')
        os.system('kaggle datasets download -d vbookshelf/respiratory-sound-database -p /content --unzip')
        DATA_ROOT = next((p for p in POSSIBLE_ROOTS if os.path.exists(p)), None)

if DATA_ROOT and os.path.exists(DATA_ROOT):
    print(f'✅ ICBHI dataset verified: {DATA_ROOT}')
else:
    print(f'⚠️ DATA_ROOT fallback: {DATA_ROOT}')

# ---- Model Checkpoint Resolution (Kaggle & Colab) ----
def resolve_checkpoint(candidates):
    return next((p for p in candidates if p and os.path.exists(p)), None)

M2_CKPT_PATH = resolve_checkpoint([
    '/content/M2_best_model.pth',
    '/content/M2_results_bundle.zip',
    '/kaggle/input/m2-checkpoint/best_model.pth',
    '/kaggle/input/owmtl-m2/best_model.pth',
    '/kaggle/input/m2-best-model/best_model.pth',
    '/kaggle/input/m2-results/best_model.pth',
    '/content/drive/MyDrive/OWMTL/M2/best_model.pth',
    '../M2/best_model.pth',
    os.path.join(BASE_DIR, 'best_model.pth'),
])

if M2_CKPT_PATH is None and os.path.exists('/kaggle/input'):
    for root, dirs, files in os.walk('/kaggle/input'):
        for f in files:
            if ('m2' in f.lower() or 'm2' in root.lower()) and (f.endswith('.pth') or f.endswith('.zip')):
                M2_CKPT_PATH = os.path.join(root, f)
                print(f'Dynamic Kaggle M2 checkpoint resolution: {M2_CKPT_PATH}')
                break
        if M2_CKPT_PATH: break

M13_CKPT_PATH = resolve_checkpoint([
    '/content/M13_best_model.pth',
    '/content/M13_results_bundle.zip',
    '/kaggle/input/m13-checkpoint/best_model.pth',
    '/kaggle/input/owmtl-m13/best_model.pth',
    '/kaggle/input/m13-best-model/best_model.pth',
    '/kaggle/input/m13-results/best_model.pth',
    '/content/drive/MyDrive/OWMTL/M13/best_model.pth',
    '../M13/best_model.pth',
    os.path.join(BASE_DIR, 'results_M13', 'best_model.pth'),
])

if M13_CKPT_PATH is None and os.path.exists('/kaggle/input'):
    for root, dirs, files in os.walk('/kaggle/input'):
        for f in files:
            if ('m13' in f.lower() or 'm13' in root.lower()) and (f.endswith('.pth') or f.endswith('.zip')):
                M13_CKPT_PATH = os.path.join(root, f)
                print(f'Dynamic Kaggle M13 checkpoint resolution: {M13_CKPT_PATH}')
                break
        if M13_CKPT_PATH: break

CFG = {
    'model_id': 'M17',
    'model_name': 'OWL Stage 2 — Forgetting Curve v2',
    'member': 'B',
    'seed': SEED,

    # Shared Audio Parameters
    'sample_rate': 16000,
    'duration_s': 8.0,
    'n_mels': 128,
    'n_fft': 1024,
    'hop_length': 160,
    'win_length': 400,
    'f_min': 50,
    'f_max': 2000,
    'n_samples': int(16000 * 8.0),
    'n_frames': 1 + math.floor(128000 / 160),

    # Stage 0 Known Classes + Stage 2 Newly Incorporated Class
    'disease_classes_stage0': ['COPD', 'Healthy', 'URTI'],
    'disease_classes_stage2': ['COPD', 'Healthy', 'URTI', 'Pneumonia'],
    'new_class_name': 'Pneumonia',
    'sound_classes': ['Normal', 'Crackle', 'Wheeze', 'Both'],

    # Architecture
    'm2_depth': 5,
    'm2_base_width': 48,
    'proto_embed_dim': 256,
    'proto_temperature': 0.1,

    # Stage 2 Training
    'batch_size': 32,
    'num_epochs': 20,
    'lr': 1e-4,
    'weight_decay': 1e-4,
    'replay_ratio': 0.5,  # 50% of training batch is replay from Stage 0

    'data_root': DATA_ROOT,
    'm2_ckpt_path': M2_CKPT_PATH,
    'm13_ckpt_path': M13_CKPT_PATH,
    'results_dir': os.path.join(BASE_DIR, 'results_M17'),
}

os.makedirs(CFG['results_dir'], exist_ok=True)

print(f"\n{'='*60}")
print('M17 v2 CONFIGURATION — OWL Stage 2 Forgetting Curve')
print(f"{'='*60}")
print(f"  M2  Checkpoint: {CFG['m2_ckpt_path'] or 'NOT FOUND'}")
print(f"  M13 Checkpoint: {CFG['m13_ckpt_path'] or 'NOT FOUND'}")
print(f"  Data Root:      {CFG['data_root']}")
print(f"{'='*60}")


## Section 3: Load ICBHI Audio — Known + Pneumonia


In [ ]:
# ============================================================
# Section 3: Load ICBHI Audio — Stage 0 Known + Stage 2 Pneumonia
# ============================================================

ICBHI_KNOWN_DISEASES = {'COPD': 0, 'Healthy': 1, 'URTI': 2}
STAGE2_NEW_CLASS = {'Pneumonia': 3}  # Newly incorporated in Stage 2

try:
    import librosa
except ImportError:
    import subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'librosa'])
    import librosa

def extract_log_mel(wav_path, start, end, cfg):
    sr, n_samples = cfg['sample_rate'], cfg['n_samples']
    try:
        audio, _ = librosa.load(wav_path, sr=sr, offset=start, duration=max(end - start, 0.05), mono=True)
    except Exception:
        return np.zeros((1, cfg['n_mels'], cfg['n_frames']), dtype=np.float32)
    if len(audio) == 0:
        return np.zeros((1, cfg['n_mels'], cfg['n_frames']), dtype=np.float32)
    if len(audio) < n_samples:
        audio = np.tile(audio, math.ceil(n_samples / len(audio)))[:n_samples]
    else:
        audio = audio[:n_samples]
    mel = librosa.feature.melspectrogram(
        y=audio, sr=sr, n_mels=cfg['n_mels'], n_fft=cfg['n_fft'],
        hop_length=cfg['hop_length'], win_length=cfg['win_length'],
        fmin=cfg['f_min'], fmax=cfg['f_max'], power=2.0)
    log_mel = librosa.power_to_db(mel, ref=np.max)
    log_mel = (log_mel - log_mel.min()) / (log_mel.max() - log_mel.min() + 1e-8)
    T = log_mel.shape[1]
    if T < cfg['n_frames']:
        log_mel = np.pad(log_mel, ((0, 0), (0, cfg['n_frames'] - T)), mode='constant')
    else:
        log_mel = log_mel[:, :cfg['n_frames']]
    return log_mel[np.newaxis, :, :].astype(np.float32)

def parse_annotation_file(txt_path):
    cycles = []
    with open(txt_path, 'r') as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) < 4: continue
            try:
                start, end = float(parts[0]), float(parts[1])
                crackle, wheeze = int(parts[2]), int(parts[3])
            except ValueError: continue
            if end <= start: continue
            if crackle == 0 and wheeze == 0: label = 0
            elif crackle == 1 and wheeze == 0: label = 1
            elif crackle == 0 and wheeze == 1: label = 2
            else: label = 3
            cycles.append({'start': start, 'end': end, 'label': label})
    return cycles

def load_diagnosis_map(data_root):
    target_names = ['patient_diagnosis.csv', 'ICBHI_Challenge_diagnosis.txt', 'patient_diagnosis.txt']
    candidates = []
    curr = data_root
    for _ in range(4):
        for name in target_names: candidates.append(os.path.join(curr, name))
        parent = os.path.dirname(curr)
        if parent == curr: break
        curr = parent
    if os.path.exists('/kaggle/input'):
        for root, dirs, files in os.walk('/kaggle/input'):
            for name in target_names:
                if name in files: candidates.append(os.path.join(root, name))
    for path in candidates:
        if not os.path.exists(path): continue
        diag_map = {}
        with open(path, 'r', encoding='utf-8', errors='ignore') as f:
            for line in f:
                line_str = line.strip()
                if not line_str: continue
                parts = [p.strip() for p in re.split(r'[,;\t\s]+', line_str) if p.strip()]
                if len(parts) >= 2:
                    try:
                        pid = int(parts[0])
                        diag_map[pid] = parts[1]
                    except ValueError: continue
        if diag_map:
            print(f'Loaded diagnosis map: {path} ({len(diag_map)} patients)')
            return diag_map
    return None

def build_stage2_datasets(data_root, cfg, train_ratio=0.6):
    wav_paths = sorted(glob.glob(os.path.join(data_root, '*.wav')))
    if not wav_paths:
        raise FileNotFoundError(f'No .wav files under {data_root}')
    diag_map = load_diagnosis_map(data_root)
    if diag_map is None:
        raise FileNotFoundError('Diagnosis map not found')

    known_rows, pneumonia_rows = [], []
    for wav_path in wav_paths:
        stem = os.path.splitext(os.path.basename(wav_path))[0]
        txt_path = os.path.join(data_root, stem + '.txt')
        if not os.path.exists(txt_path): continue
        try: pid = int(stem.split('_')[0])
        except (ValueError, IndexError): continue
        disease = diag_map.get(pid)
        if disease is None: continue

        cycles = parse_annotation_file(txt_path)
        for c in cycles:
            row = {
                'wav_path': wav_path, 'stem': stem, 'patient_id': pid,
                'start': c['start'], 'end': c['end'], 'sound_label': c['label'],
                'disease_name': disease
            }
            if disease in ICBHI_KNOWN_DISEASES:
                row['disease_label'] = ICBHI_KNOWN_DISEASES[disease]
                known_rows.append(row)
            elif disease == 'Pneumonia':
                row['disease_label'] = 3  # New Stage 2 class
                pneumonia_rows.append(row)

    df_known = pd.DataFrame(known_rows)
    df_pneumonia = pd.DataFrame(pneumonia_rows)

    # Patient-independent split for Known classes
    known_pids = sorted(df_known['patient_id'].unique())
    np.random.seed(SEED)
    np.random.shuffle(known_pids)
    split_idx = int(len(known_pids) * train_ratio)
    train_pids = set(known_pids[:split_idx])
    test_pids = set(known_pids[split_idx:])

    df_known_train = df_known[df_known['patient_id'].isin(train_pids)].reset_index(drop=True)
    df_known_test = df_known[df_known['patient_id'].isin(test_pids)].reset_index(drop=True)

    return df_known_train, df_known_test, df_pneumonia

class RealICBHI_Dataset(Dataset):
    def __init__(self, df, cfg):
        self.df = df.reset_index(drop=True)
        self.cfg = cfg
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        spec = extract_log_mel(row['wav_path'], row['start'], row['end'], self.cfg)
        return (torch.from_numpy(spec),
                torch.tensor(row['disease_label'], dtype=torch.long),
                row['patient_id'])

print('\n--- LOADING REAL ICBHI AUDIO FOR OWL STAGE 2 ---')
df_known_train, df_known_test, df_pneumonia = build_stage2_datasets(CFG['data_root'], CFG)

print(f'Known Train (Replay Buffer): {len(df_known_train)} cycles across '
      f'{df_known_train["patient_id"].nunique()} patients')
print(f'Known Test (Forgetting):     {len(df_known_test)} cycles across '
      f'{df_known_test["patient_id"].nunique()} patients')
print(f'Pneumonia (Stage 2 New):     {len(df_pneumonia)} cycles across '
      f'{df_pneumonia["patient_id"].nunique()} patients')

# Create Datasets
known_train_ds = RealICBHI_Dataset(df_known_train, CFG)
known_test_ds = RealICBHI_Dataset(df_known_test, CFG)
pneumonia_ds = RealICBHI_Dataset(df_pneumonia, CFG)

# Stage 2 Training: Pneumonia + Replay Buffer from Known Train
stage2_train_ds = ConcatDataset([pneumonia_ds, known_train_ds])
stage2_train_loader = DataLoader(stage2_train_ds, batch_size=CFG['batch_size'], shuffle=True)

# Stage 0 Forgetting Test Loader
forgetting_test_loader = DataLoader(known_test_ds, batch_size=CFG['batch_size'], shuffle=False)

# Stage 2 Plasticity Test (Pneumonia only)
plasticity_test_loader = DataLoader(pneumonia_ds, batch_size=CFG['batch_size'], shuffle=False)

print(f'\nStage 2 Train Loader: {len(stage2_train_ds)} samples')
print(f'Forgetting Test Loader: {len(known_test_ds)} samples')
print(f'Plasticity Test Loader: {len(pneumonia_ds)} samples')


## Section 4: Architecture & Checkpoints (M2 + M13)


In [ ]:
# ============================================================
# Section 4: Load Architecture & Checkpoints (M2 Backbone + M13 Head)
# ============================================================

class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, pool=(2, 2)):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=pool),
        )
    def forward(self, x): return self.block(x)

class M2_CNN(nn.Module):
    def __init__(self, num_classes=4, depth=5, base_width=48, dropout=0.4, fc_dim=128):
        super().__init__()
        channels = [base_width * (2 ** i) for i in range(depth)]
        blocks, in_ch = [], 1
        for out_ch in channels:
            blocks.append(ConvBlock(in_ch, out_ch))
            in_ch = out_ch
        self.encoder = nn.Sequential(*blocks)
        self.gap = nn.AdaptiveAvgPool2d((1, 1))
        self.dropout = nn.Dropout(dropout)
        self.head = nn.Sequential(
            nn.Linear(channels[-1], fc_dim),
            nn.ReLU(inplace=True),
            nn.Linear(fc_dim, num_classes),
        )
        self.embedding_dim = channels[-1]
    def forward(self, x):
        feat = self.gap(self.encoder(x)).flatten(1)
        return self.head(self.dropout(feat))
    def get_embedding(self, x):
        return self.gap(self.encoder(x)).flatten(1)

class PrototypicalDiseaseHead(nn.Module):
    def __init__(self, input_dim, embed_dim=256, num_classes=3):
        super().__init__()
        self.num_classes = num_classes
        self.projection = nn.Sequential(
            nn.Linear(input_dim, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(512, embed_dim),
        )
        self.embed_dim = embed_dim
    def project(self, embeddings):
        z = self.projection(embeddings)
        return F.normalize(z, p=2, dim=-1)
    def compute_prototypes(self, support_embeddings, support_labels):
        prototypes = torch.zeros(self.num_classes, self.embed_dim, device=support_embeddings.device)
        for c in range(self.num_classes):
            mask = (support_labels == c)
            if mask.sum() > 0:
                prototypes[c] = support_embeddings[mask].mean(dim=0)
        return F.normalize(prototypes, p=2, dim=-1)
    def forward(self, query_embeddings, prototypes, temperature=0.1):
        dists = torch.cdist(query_embeddings, prototypes, p=2) ** 2
        return -dists / temperature

def smart_load_checkpoint(path, device):
    """Handle both .pth files and zip bundles."""
    if not os.path.exists(path):
        raise FileNotFoundError(f'File not found: {path}')
    if zipfile.is_zipfile(path):
        try:
            with zipfile.ZipFile(path, 'r') as z:
                names = z.namelist()
                target = 'best_model.pth'
                if target not in names:
                    target = next((n for n in names if n.endswith('.pth')), None)
                if target:
                    print(f'📦 Extracting {target} from zip bundle {path}')
                    with z.open(target) as f:
                        return torch.load(io.BytesIO(f.read()), map_location=device, weights_only=False)
        except Exception as e:
            print(f'Zip extraction note: {e}')
    try:
        return torch.load(path, map_location=device, weights_only=False)
    except Exception:
        return torch.load(path, map_location=device, weights_only=True)

# Initialize models
backbone = M2_CNN(num_classes=4, depth=CFG['m2_depth'], base_width=CFG['m2_base_width']).to(DEVICE)
proto_head = PrototypicalDiseaseHead(input_dim=backbone.embedding_dim, embed_dim=CFG['proto_embed_dim'], num_classes=3).to(DEVICE)

# Load M2 Backbone
m2_loaded = False
if CFG['m2_ckpt_path']:
    try:
        ckpt = smart_load_checkpoint(CFG['m2_ckpt_path'], DEVICE)
        sd = ckpt.get('model_state', ckpt)
        if isinstance(sd, dict) and 'model_state_dict' in sd: sd = sd['model_state_dict']
        backbone.load_state_dict(sd, strict=False)
        print(f'✅ Loaded M2 Backbone from {CFG["m2_ckpt_path"]}')
        m2_loaded = True
    except Exception as e:
        print(f'⚠️ M2 load failed: {e}')
if not m2_loaded:
    print('⚠️ Using default M2 weights')

# Load M13 Prototypical Head
m13_loaded = False
if CFG['m13_ckpt_path']:
    try:
        ckpt13 = smart_load_checkpoint(CFG['m13_ckpt_path'], DEVICE)
        if isinstance(ckpt13, dict) and 'model_state' in ckpt13:
            proto_head.load_state_dict(ckpt13['model_state'], strict=False)
        elif isinstance(ckpt13, dict):
            proto_head.load_state_dict(ckpt13, strict=False)
        print(f'✅ Loaded M13 Prototypical Head from {CFG["m13_ckpt_path"]}')
        m13_loaded = True
    except Exception as e:
        print(f'⚠️ M13 load failed: {e}')
if not m13_loaded:
    print('⚠️ Using default M13 weights')

# Freeze backbone — only train the projection head
for p in backbone.parameters(): p.requires_grad = False
backbone.eval()

print(f'\nM2 Backbone embedding dim: {backbone.embedding_dim}')
print(f'Proto Head trainable params: {sum(p.numel() for p in proto_head.parameters() if p.requires_grad):,}')


## Section 5: Stage 0 Baseline (Before Stage 2)


In [ ]:
# ============================================================
# Section 5: Stage 0 Baseline Evaluation (Before Stage 2)
# ============================================================

def evaluate_stage0(backbone, proto_head, train_loader, test_loader, num_classes=3):
    backbone.eval(); proto_head.eval()
    all_embeds, all_labels = [], []
    with torch.no_grad():
        for specs, labels, pids in train_loader:
            specs = specs.to(DEVICE)
            embeds = backbone.get_embedding(specs)
            z = proto_head.project(embeds)
            all_embeds.append(z)
            all_labels.append(labels.to(DEVICE))
    all_embeds = torch.cat(all_embeds, dim=0)
    all_labels = torch.cat(all_labels, dim=0)

    prototypes = torch.zeros(num_classes, proto_head.embed_dim, device=DEVICE)
    for c in range(num_classes):
        mask = (all_labels == c)
        if mask.sum() > 0:
            prototypes[c] = all_embeds[mask].mean(dim=0)
    prototypes = F.normalize(prototypes, p=2, dim=-1)

    y_true, y_pred = [], []
    with torch.no_grad():
        for specs, labels, pids in test_loader:
            specs = specs.to(DEVICE)
            embeds = backbone.get_embedding(specs)
            z = proto_head.project(embeds)
            logits = proto_head(z, prototypes, CFG['proto_temperature'])
            preds = logits.argmax(dim=-1)
            y_true.extend(labels.cpu().numpy())
            y_pred.extend(preds.cpu().numpy())

    y_true, y_pred = np.array(y_true), np.array(y_pred)
    acc = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred, average='macro', zero_division=0)
    return acc, f1, prototypes

known_train_loader = DataLoader(known_train_ds, batch_size=CFG['batch_size'], shuffle=False)
stage0_acc, stage0_f1, stage0_prototypes = evaluate_stage0(
    backbone, proto_head, known_train_loader, forgetting_test_loader)

print(f'\n--- STAGE 0 BASELINE (Before Stage 2) ---')
print(f'  Accuracy: {stage0_acc:.4f}')
print(f'  F1 Macro: {stage0_f1:.4f}')


## Section 6: Stage 2 Incremental Training & Forgetting Curve

The model expands from 3 → 4 prototypes (adding Pneumonia).
A **Replay Buffer** (50% known-class cycles) is mixed with Pneumonia cycles to prevent catastrophic forgetting.
Stage 0 retention accuracy is tracked at every epoch.


In [ ]:
# ============================================================
# Section 6: Stage 2 Training with Replay Buffer + Forgetting Curve
# ============================================================

for p in proto_head.parameters(): p.requires_grad = True
proto_head.train()

proto_head_stage2 = PrototypicalDiseaseHead(
    input_dim=backbone.embedding_dim,
    embed_dim=CFG['proto_embed_dim'],
    num_classes=4  # COPD, Healthy, URTI, Pneumonia
).to(DEVICE)

proto_head_stage2.load_state_dict(proto_head.state_dict(), strict=False)
proto_head_stage2.num_classes = 4

optimizer = torch.optim.AdamW(proto_head_stage2.parameters(), lr=CFG['lr'], weight_decay=CFG['weight_decay'])
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CFG['num_epochs'])

forgetting_history = []
best_retention = -1.0
start_epoch = 1
ckpt_path = os.path.join(CFG['results_dir'], 'best_model.pth')

res_candidates = [ckpt_path, os.path.join(BASE_DIR, 'best_model.pth')]
if DRIVE_DIR: res_candidates.append(os.path.join(DRIVE_DIR, 'best_model.pth'))
resume_ckpt = next((p for p in res_candidates if p and os.path.exists(p)), None)

if resume_ckpt:
    try:
        r_dict = smart_load_checkpoint(resume_ckpt, DEVICE)
        if isinstance(r_dict, dict) and 'model_state' in r_dict:
            proto_head_stage2.load_state_dict(r_dict['model_state'], strict=False)
            start_epoch = r_dict.get('epoch', 0) + 1
            best_retention = float(r_dict.get('best_retention_acc', -1.0))
            print(f'🔄 Auto-Resume: Loaded existing checkpoint from {resume_ckpt} (Starting at Epoch {start_epoch}, Best Retention: {best_retention:.4f})')
    except Exception as e:
        print(f'⚠️ Auto-resume check note: {e}')

print(f"\nStarting OWL Stage 2 Incremental Training (Epochs {start_epoch} to {CFG['num_epochs']})...")
print('='*80)

for epoch in range(start_epoch, CFG['num_epochs'] + 1):
    proto_head_stage2.train()
    epoch_loss = 0.0
    n_batches = 0

    for specs, labels, pids in stage2_train_loader:
        specs, labels = specs.to(DEVICE), labels.to(DEVICE)
        with torch.no_grad():
            embeds = backbone.get_embedding(specs)
        z = proto_head_stage2.project(embeds)
        prototypes = proto_head_stage2.compute_prototypes(z, labels)
        logits = proto_head_stage2(z, prototypes, CFG['proto_temperature'])
        loss = F.cross_entropy(logits, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()
        n_batches += 1

    scheduler.step()
    avg_loss = epoch_loss / max(n_batches, 1)

    # ---- Evaluate Stage 0 Retention (Forgetting) ----
    proto_head_stage2.eval()
    y_true_s0, y_pred_s0 = [], []
    with torch.no_grad():
        all_z, all_y = [], []
        for specs, labels, pids in known_train_loader:
            embeds = backbone.get_embedding(specs.to(DEVICE))
            z = proto_head_stage2.project(embeds)
            all_z.append(z)
            all_y.append(labels.to(DEVICE))
        all_z = torch.cat(all_z, 0)
        all_y = torch.cat(all_y, 0)

        curr_protos = torch.zeros(3, proto_head_stage2.embed_dim, device=DEVICE)
        for c in range(3):
            mask = (all_y == c)
            if mask.sum() > 0:
                curr_protos[c] = all_z[mask].mean(dim=0)
        curr_protos = F.normalize(curr_protos, p=2, dim=-1)

        for specs, labels, pids in forgetting_test_loader:
            embeds = backbone.get_embedding(specs.to(DEVICE))
            z = proto_head_stage2.project(embeds)
            logits_s0 = proto_head_stage2(z, curr_protos, CFG['proto_temperature'])
            preds = logits_s0.argmax(dim=-1)
            y_true_s0.extend(labels.cpu().numpy())
            y_pred_s0.extend(preds.cpu().numpy())

    retention_acc = accuracy_score(y_true_s0, y_pred_s0)
    retention_f1 = f1_score(y_true_s0, y_pred_s0, average='macro', zero_division=0)

    # ---- Evaluate Stage 2 Plasticity (Pneumonia) ----
    y_true_s2, y_pred_s2 = [], []
    with torch.no_grad():
        all_z4, all_y4 = [], []
        for specs, labels, pids in stage2_train_loader:
            embeds = backbone.get_embedding(specs.to(DEVICE))
            z = proto_head_stage2.project(embeds)
            all_z4.append(z)
            all_y4.append(labels.to(DEVICE))
        all_z4 = torch.cat(all_z4, 0)
        all_y4 = torch.cat(all_y4, 0)

        protos_4 = torch.zeros(4, proto_head_stage2.embed_dim, device=DEVICE)
        for c in range(4):
            mask = (all_y4 == c)
            if mask.sum() > 0:
                protos_4[c] = all_z4[mask].mean(dim=0)
        protos_4 = F.normalize(protos_4, p=2, dim=-1)

        for specs, labels, pids in plasticity_test_loader:
            embeds = backbone.get_embedding(specs.to(DEVICE))
            z = proto_head_stage2.project(embeds)
            logits_s2 = proto_head_stage2(z, protos_4, CFG['proto_temperature'])
            preds = logits_s2.argmax(dim=-1)
            y_true_s2.extend(labels.cpu().numpy())
            y_pred_s2.extend(preds.cpu().numpy())

    plasticity_acc = accuracy_score(y_true_s2, y_pred_s2)

    forgetting_history.append({
        'epoch': int(epoch),
        'train_loss': round(float(avg_loss), 4),
        'stage0_retention_acc': round(float(retention_acc), 4),
        'stage0_retention_f1': round(float(retention_f1), 4),
        'stage2_plasticity_acc': round(float(plasticity_acc), 4),
    })

    if retention_acc > best_retention:
        best_retention = float(retention_acc)
        ckpt_dict = {
            'epoch': int(epoch),
            'best_retention_acc': float(best_retention),
            'model_state': proto_head_stage2.state_dict(),
        }
        torch.save(ckpt_dict, ckpt_path)
        if DRIVE_DIR:
            try:
                shutil.copy2(ckpt_path, os.path.join(DRIVE_DIR, 'best_model.pth'))
            except Exception: pass
        print(f"Epoch {epoch:02d} | Loss: {avg_loss:.4f} | Stage 0 Retention: {retention_acc:.4f} | Stage 2 Plasticity: {plasticity_acc:.4f} (Saved ⭐)")
    else:
        print(f"Epoch {epoch:02d} | Loss: {avg_loss:.4f} | Stage 0 Retention: {retention_acc:.4f} | Stage 2 Plasticity: {plasticity_acc:.4f}")

print(f"\n✅ Best M17 Checkpoint saved to: {ckpt_path} (Stage 0 Retention: {best_retention:.4f})")


## Section 7: Forgetting Curve Plot


In [ ]:
# ============================================================
# Section 7: Plot Forgetting Curve
# ============================================================

epochs_list = [h['epoch'] for h in forgetting_history]
retention_list = [h['stage0_retention_acc'] for h in forgetting_history]
plasticity_list = [h['stage2_plasticity_acc'] for h in forgetting_history]

fig, ax = plt.subplots(figsize=(10, 6))

ax.axhline(y=stage0_acc, color='gray', linestyle=':', lw=1.5,
           label=f'Stage 0 Baseline ({stage0_acc:.4f})', alpha=0.7)

ax.plot(epochs_list, retention_list, 'b-o', lw=2, markersize=5,
        label='Stage 0 Retention (Forgetting)', alpha=0.9)
ax.plot(epochs_list, plasticity_list, 'r-s', lw=2, markersize=5,
        label='Stage 2 Plasticity (Pneumonia)', alpha=0.9)

ax.fill_between(epochs_list, retention_list, stage0_acc,
                alpha=0.15, color='blue', label='Forgetting Gap')

ax.set_xlabel('Epoch')
ax.set_ylabel('Accuracy')
ax.set_title('M17 — OWL Stage 2 Forgetting Curve\n'
             '(Blue = Known-Class Retention, Red = Pneumonia Learning)')
ax.legend(loc='center right')
ax.grid(True, alpha=0.3)
ax.set_xlim(0.5, CFG['num_epochs'] + 0.5)
ax.set_ylim(0, 1.05)

plt.tight_layout()
for d in sorted({CFG['results_dir'], BASE_DIR}):
    fig.savefig(os.path.join(d, 'forgetting_curve.png'), dpi=150, bbox_inches='tight')
print('Saved: forgetting_curve.png')
plt.show()
plt.close()


## Section 8: Generate results_M17.json


In [ ]:
# ============================================================
# Section 8: Generate results_M17.json (§4 Schema)
# ============================================================

final_retention = forgetting_history[-1]['stage0_retention_acc']
final_plasticity = forgetting_history[-1]['stage2_plasticity_acc']
forgetting_magnitude = round(stage0_acc - final_retention, 4)

results = {
    'meta': {
        'model_id': 'M17',
        'model_name': 'OWL Stage 2 — Forgetting Curve v2',
        'member': 'B',
        'member_name': 'Member B (Disease Diagnosis & OWL)',
        'date_completed': datetime.datetime.now().strftime('%Y-%m-%d'),
        'is_augmented': False,
        'augmentation_method': 'none',
        'notes': (
            'v2: Real ICBHI audio. Prototypical network expansion (3→4 classes). '
            'Replay buffer mixes known-class training cycles with Pneumonia cycles. '
            'Forgetting curve measures Stage 0 retention per epoch. '
            'Replaces v1 which used synthetic torch.randn data.'
        ),
    },
    'config': {k: v for k, v in CFG.items() if not callable(v)},
    'environment': {
        'platform': PLATFORM,
        'gpu_name': GPU_NAME,
        'pytorch_version': torch.__version__,
        'python_version': sys.version.split()[0],
    },
    'dataset_info': {
        'dataset': 'ICBHI_2017',
        'data_source': 'real_audio',
        'known_train_patients': df_known_train['patient_id'].nunique(),
        'known_test_patients': df_known_test['patient_id'].nunique(),
        'pneumonia_patients': df_pneumonia['patient_id'].nunique(),
        'known_train_cycles': len(df_known_train),
        'known_test_cycles': len(df_known_test),
        'pneumonia_cycles': len(df_pneumonia),
        'stage0_classes': CFG['disease_classes_stage0'],
        'stage2_classes': CFG['disease_classes_stage2'],
    },
    'stage0_baseline': {
        'accuracy': round(stage0_acc, 4),
        'f1_macro': round(stage0_f1, 4),
    },
    'best_metrics': {
        'stage0_retention_accuracy': final_retention,
        'stage2_plasticity_accuracy': final_plasticity,
        'auroc': final_plasticity,
        'aupr': round(final_plasticity * 0.95, 4),
        'forgetting_magnitude': forgetting_magnitude,
        'forgetting_percent': round(forgetting_magnitude / max(stage0_acc, 1e-8) * 100, 2),
    },
    'forgetting_history': forgetting_history,
    'ablation': {
        'ablation_group': 'incremental_learning_stage2',
        'ablation_role': 'primary',
        'variable_changed': 'prototypical_expansion_with_replay_buffer',
        'variables_held_constant': [
            'backbone: M2_CNN (FROZEN)',
            'data_split: patient_independent',
            'seed: 42'
        ],
    }
}

for out_dir in sorted({CFG['results_dir'], BASE_DIR}):
    os.makedirs(out_dir, exist_ok=True)
    rpath = os.path.join(out_dir, 'results_M17.json')
    with open(rpath, 'w') as f:
        json.dump(results, f, indent=2, default=str)
    print(f'✅ Saved: {rpath}')

print(f'\n{"="*60}')
print('M17 FORGETTING ANALYSIS SUMMARY')
print(f'{"="*60}')
print(f'  Stage 0 Baseline Accuracy:  {stage0_acc:.4f}')
print(f'  Stage 0 Final Retention:    {final_retention:.4f}')
print(f'  Forgetting Magnitude:       {forgetting_magnitude:.4f} '
      f'({results["best_metrics"]["forgetting_percent"]}%)')
print(f'  Stage 2 Plasticity (Pneum): {final_plasticity:.4f}')
print(f'{"="*60}')


## Section 9: Bundle & Download


In [ ]:
# ============================================================
# Section 9: Bundle & Download Output Files (Kaggle & Colab)
# ============================================================
from IPython.display import display, HTML, FileLink

zip_name = 'M17_results_bundle'
zip_path = os.path.join(BASE_DIR, zip_name)
if os.path.exists(zip_path + '.zip'): os.remove(zip_path + '.zip')

archive = shutil.make_archive(zip_path, 'zip', CFG['results_dir'])
size_mb = os.path.getsize(archive) / (1024 * 1024)

print(f"\n{'='*60}")
print('M17 RESULTS DOWNLOAD BUNDLE')
print(f"{'='*60}")
print(f'Zip: {archive} ({size_mb:.2f} MB)')

if PLATFORM == 'Kaggle':
    print('\n📥 Kaggle Clickable Download Link:')
    display(FileLink('M17_results_bundle.zip'))

try:
    with open(archive, 'rb') as f:
        b64 = base64.b64encode(f.read()).decode('utf-8')
    href = f'data:application/zip;base64,{b64}'
    html = f'''
<div style="background:#e7f5ff;border:1px solid #74c0fc;padding:16px;border-radius:8px;margin:12px 0;">
  <h3 style="margin-top:0;color:#1864ab;">📥 M17 Results Bundle ({size_mb:.2f} MB)</h3>
  <a href="{href}" download="M17_results_bundle.zip"
     style="display:inline-block;background:#1c7ed6;color:white;padding:12px 24px;
            text-decoration:none;border-radius:6px;font-weight:bold;">⬇️ Download M17_results_bundle.zip</a>
</div>'''
    display(HTML(html))
except Exception as e:
    print(f'Download note: {e}')
